In [5]:
from gurobipy import *

In [7]:
# ---------------------------------------------------------------------------
# INITIALIZATION
# intialized set of potential facility locations noFacs_, set of customers locations noCusts_ , 
#   matrix a_ij which is 1 if facility location i can cover customer location j and 
#   0 otherwise as aMatrix_. 
# demands j is population of customer location j demands_, and 
#   bound_ is the limit of the total number of facilities we can open
# ---------------------------------------------------------------------------

def initializeData():
    noFacs_ = 5
    noCusts_ = 10
    aMatrix_ = [ [ 0 for j in range( noCusts_ ) ] for i in range( noFacs_ ) ]
    aMatrix_[ 0 ] = [ 1, 0, 1, 0, 0, 0, 0, 0, 1, 0 ]
    aMatrix_[ 1 ] = [ 0, 1, 0, 0, 0, 0, 0, 1, 0, 1 ]
    aMatrix_[ 2 ] = [ 0, 0, 0, 0, 1, 0, 1, 0, 1, 0 ]
    aMatrix_[ 3 ] = [ 0, 0, 1, 1, 0, 0, 0, 0, 0, 1 ]
    aMatrix_[ 4 ] = [ 0, 0, 0, 0, 1, 1, 1, 0, 0, 0 ]
    demands_ = [ 152, 312, 243, 245, 141, 165, 456, 362, 126, 353 ]
    bound_ = 2
    return ( noFacs_ , noCusts_ , aMatrix_, demands_ , bound_ )

In [ ]:
# ---------------------------------------------------------------------------
# CONSTRUCT VARIABLES
# constructed binary variables x_i for each potential facility location i and
#   binary variables y_j for each customer location j 
# then update model with the variables
# ---------------------------------------------------------------------------

def constructVars():
    global noFacs , noCusts , ipModel , xVars , yVars
    for i in range( noFacs ):
        var = ipModel.addVar( vtype = GRB.BINARY , name = "x_" + str(i) )
        xVars[ i ] = var
    for j in range( noCusts ):
        var = ipModel.addVar( vtype = GRB.BINARY , name = "y_" + str(j) )
        yVars[ j ] = var
    ipModel.update()

In [ ]:
# ---------------------------------------------------------------------------
# CONSTRUCT OBJECTION FUNCTION
# constructed objective function to maximize the total population covered 
#   by the selected facilities
# did this by first creating empty expression for objective function, then 
#   adding the product of the population of each customer location and 
#   the corresponding binary variable y_j
# then update the model with the objective function
# ---------------------------------------------------------------------------

def constructObj():
    global noCusts , yVars , demands
    objExpr = LinExpr()
    for j in range( noCusts ):
        objExpr += demands[ j ] * yVars[ j ]
    ipModel.setObjective( objExpr , GRB.MAXIMIZE )
    ipModel.update()

In [14]:
# ---------------------------------------------------------------------------
# CONSTRUCT CONSTRAINTS
# constructed two sets of contraints.
#   the first being the coverage constraints which ensure that if 
#       a customer location j is covered, then 
#       at least one of the facility locations i that can cover must be selected
#   then update the model with the constraints 
#   the second being the limit constraint which ensures that the total number of
#      selected facility locations does not exceed the limit bound. 
# then update the model with the constraints 
# ---------------------------------------------------------------------------

def constructConsts():
    global noFacs , noCusts, ipModel , xVars , yVars , aMatrix , bound
    for j in range( noCusts ):
        constrExpr = LinExpr()
        constrExpr += 1.0 * yVars[ j ]
        for i in range( noFacs ):
            constrExpr += - aMatrix[ i ][ j ] * xVars[ i ]
        ipModel.addLConstr( lhs = constrExpr , sense = GRB.LESS_EQUAL , rhs = 0 , name = "cover_" + str(j) )
    constrExpr = LinExpr()
    for i in range( noFacs ):
        constrExpr += 1.0 * xVars[ i ]
        ipModel.addLConstr( lhs = constrExpr , sense = GRB.LESS_EQUAL , rhs = bound , name = "limit" )
    ipModel.update()

In [15]:
# ---------------------------------------------------------------------------
# PRINT SOLUTIONS
# print optimal objective value and the values of the decision variables x_i and y_j
# ---------------------------------------------------------------------------
  
def printSolution():
    global noFacs , noCusts , xVars , yVars , ipModel
    print( "Opt. Obj." + str( ipModel.ObjVal ) )
    for i in range( noFacs ):
        print( xVars[ i ].varName + " " + str( xVars[ i ].x ) )
    for j in range( noCusts ):
        print( yVars[ j ].varName + " " + str( yVars[ j ].x ) )

In [16]:
# ---------------------------------------------------------------------------
# RUNNING COMPUTATION
# returns the initialized data, 
#   constructs the model, 
#   writes the model to a file, 
#   optimizes the model, and 
#   prints the solution
# ---------------------------------------------------------------------------

( noFacs , noCusts , aMatrix , demands , bound ) = initializeData()
ipModel = Model( "set_covering" )
xVars = [ 0 for i in range( noFacs ) ]
yVars = [ 0 for j in range( noCusts ) ]
constructVars()
constructObj()
constructConsts()
ipModel.write( "october_29_2026.lp")

Set parameter Username
Set parameter LicenseID to value 2878462
Academic license - for non-commercial use only - expires 2027-10-26


In [ ]:
# ---------------------------------------------------------------------------
# RUNNING COMPUTATION
# once the .lp file looks good, 
#   optimize the model, and 
#   print the solution
# ---------------------------------------------------------------------------

ipModel.optimize()
printSolution()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G720)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 15 rows, 15 columns and 40 nonzeros (Max)
Model fingerprint: 0xeecdbe71
Model has 10 linear objective coefficients
Variable types: 0 continuous, 15 integer (15 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 5e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]

Found heuristic solution: objective -0.0000000
Presolve removed 10 rows and 6 columns
Presolve time: 0.00s
Presolved: 5 rows, 9 columns, 17 nonzeros
Variable types: 0 continuous, 9 integer (9 binary)
Found heuristic solution: objective 1548.0000000

Root relaxation: objective 1.789000e+03, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Best